# CAPE demo

This notebook runs the native Rust/PyO3 CAPE extraction path on the official CAPE `Data/seq_example` frame from https://github.com/pedropro/CAPE/tree/master/Data/seq_example. Run `uv sync --all-extras` and `uv run maturin develop --release` before executing it.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON_DIR = PROJECT_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
from cape_demo.official_data import load_official_seq_example
from cape_demo.visualization import save_demo_figure, overlay_labels

scene = load_official_seq_example(PROJECT_ROOT / "data" / "seq_example")
print(f"source={scene.source_dir}")
print(scene.depth.shape, scene.depth.dtype, scene.depth.min(), scene.depth.max())
print(scene.fx, scene.fy, scene.cx, scene.cy)


In [ ]:
from cape_demo.extract import extract_frame

result = extract_frame(scene)
print("engine=cape_pyo3")
print(f"planes={len(result['planes'])}, cylinders={len(result['cylinders'])}")
print(result["stats"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

preview_path = save_demo_figure(scene, result, ARTIFACTS / "demo_preview.png")

fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(1, 3, 1)
ax1.imshow(scene.depth, cmap="viridis")
ax1.set_title("Depth")
ax1.axis("off")

ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(np.asarray(result["labels"]), cmap="tab20")
ax2.set_title("Labels")
ax2.axis("off")

ax3 = fig.add_subplot(1, 3, 3)
ax3.imshow(overlay_labels(scene.rgb, np.asarray(result["labels"])))
ax3.set_title("Overlay")
ax3.axis("off")
plt.tight_layout()


In [ ]:
from pprint import pprint
print("Planes")
pprint(result["planes"])
print("Cylinders")
pprint(result["cylinders"])